# 26종 전체 데이터 1차 학습 — min100

학습 3,128장 · 검증 204장 · 테스트 205장. YOLO11n 26종 장비 탐지 모델, T4 GPU, 최대 100 epochs, patience 20, 640px, batch 16.

## 2026-11-01 재학습
이전 실행은 70회차에서 조기 종료했지만 Colab 임시 디스크의 최종 모델을 잃었습니다. Drive에 남아 있는 초기 체크포인트의 모델·옵티마이저 상태에서 이어서 학습합니다.

**이번에는 체크포인트와 평가 결과를 Google Drive에 직접 저장합니다.** 실행 전 Drive 여유 공간 약 2.9GB 확인. 저장 폴더: `SKKU_1398_Fitness_ML/03_training_runs/equipment26_min100_retrain_<시각>/`. 매 회차 `fit/weights/last.pt`, 최고 성능 `fit/weights/best.pt`, 진행 상태 `status.json`을 저장합니다.

마지막 셀에 실시간 학습 로그가 표시됩니다. 학습 중 셀을 다시 실행하지 마세요. 완료 후 테스트 205장 평가와 장비별 결과를 생성합니다. 일부 장비의 테스트 표본이 적으므로 현장 성능은 별도로 확인해야 합니다.

In [ ]:
from google.colab import drive
import subprocess, sys, torch
assert torch.cuda.is_available(), '런타임 유형을 GPU로 변경하세요.'
drive.mount('/content/drive')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.194'], check=True)


In [ ]:
%%writefile /content/prepare_min100.py
"""Reconstruct all 26 classes from the existing Drive archives plus this small update."""
import argparse,hashlib,shutil,subprocess,sys,zipfile
from pathlib import Path
DRIVE=Path('/content/drive/MyDrive/SKKU_1398_Fitness_ML/02_dataset/SKKU1398_equipment_v1/yolo')
def sha(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
 return h.hexdigest()
def main():
 p=argparse.ArgumentParser();p.add_argument('--base-zip',type=Path,default=DRIVE/'equipment24_expanded_min100_20261031.zip');p.add_argument('--previous-zip',type=Path,default=DRIVE/'equipment26_additions_labeled_20261101.zip');p.add_argument('--update-zip',type=Path,default=DRIVE/'equipment26_min100_v2_update.zip');p.add_argument('--output',type=Path,default=Path('/content/equipment26_min100_v2'));a=p.parse_args()
 for f,h in [(a.base_zip,'587491a3e2d6ea98310f9c7de489a12b3a139c94de74b765499cd77e92a24f41'),(a.previous_zip,'7357a197939ab83a600b1a8bffac2031ae131b929335af65cef6631e1fc983b5')]:
  if not f.is_file():raise SystemExit(f'Missing required archive: {f}')
  if sha(f)!=h:raise SystemExit(f'Unexpected archive contents: {f}')
 if not a.update_zip.is_file():raise SystemExit(f'Missing update: {a.update_zip}')
 if a.output.exists() and any(a.output.iterdir()):raise SystemExit('Output must be empty. Choose a new --output folder; existing data is preserved.')
 a.output.mkdir(parents=True,exist_ok=True)
 for archive,rootname,images_only in [(a.base_zip,None,True),(a.previous_zip,'equipment26_20261101',True),(a.update_zip,'equipment26_min100_v2',False)]:
  with zipfile.ZipFile(archive) as z:
   for item in z.infolist():
    parts=Path(item.filename).parts
    if item.is_dir() or len(parts)<2 or (rootname and parts[0]!=rootname):continue
    rel=Path(*parts[1:])
    if images_only and rel.parts[0] not in ('images','labels'):continue
    dest=a.output/rel
    if not dest.resolve().is_relative_to(a.output.resolve()):raise ValueError('Unsafe archive path')
    dest.parent.mkdir(parents=True,exist_ok=True)
    with z.open(item) as src,dest.open('wb') as dst:shutil.copyfileobj(src,dst)
 subprocess.run([sys.executable,str(a.output/'verify_dataset.py')],check=True)
 print('Verified dataset:',a.output.resolve())
 print('YOLO data:',(a.output/'data.yaml').resolve())
 print('Model training has not been started by this script.')
if __name__=='__main__':main()


In [ ]:
%%writefile /content/train_min100.py
"""Train the verified 26-class min100 dataset; no existing runs are overwritten."""
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, shutil, subprocess, sys, traceback, gc

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()

def write_json(path, value):
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding='utf-8')
    temp.replace(path)

def main():
    import torch, yaml, ultralytics
    from ultralytics import YOLO, settings
    root=Path('/content/drive/MyDrive/SKKU_1398_Fitness_ML')
    data=Path('/content/equipment26_min100_v2')
    expected='3faea06d469f471e85cd2dd976a55049cafcb9db5e4d96809e42a923cb754706'
    assert torch.cuda.is_available(),'Colab GPU 런타임이 필요합니다.'
    assert sha(data/'manifest.json')==expected,'Dataset version mismatch'
    subprocess.run([sys.executable,str(data/'verify_dataset.py')],check=True)
    summary=json.loads((data/'summary.json').read_text())
    mapping=json.loads((data/'class_mapping.json').read_text())
    assert summary['images']==3537 and summary['splits']=={'train':3128,'val':204,'test':205}
    names={m['yolo_id']:m['name'] for m in mapping}
    previous='equipment26_yolo11n_20261031T125943Z'
    candidates=[root/'04_exports_and_reports'/previous/'best.pt',root/'03_training_runs'/previous/'fit/weights/best.pt']
    source=next((p for p in candidates if p.is_file()),None)
    assert source is not None,'기존 26종 best.pt를 찾을 수 없습니다.'
    model=YOLO(str(source))
    assert model.names==names, f'Checkpoint class mapping mismatch: {model.names}'
    settings.update({k:False for k in ('wandb','comet','mlflow','clearml','neptune','hub') if k in settings})
    resume=os.environ.get('EQUIPMENT26_MIN100_RESUME_RUN','').strip()
    if resume:
        run=Path(resume)
        assert run.parent==root/'03_training_runs'
        saved=json.loads((run/'status.json').read_text())
        assert saved['dataset_manifest_sha256']==expected and saved['state']!='completed'
        checkpoint=run/'fit/weights/last.pt'
        assert checkpoint.is_file()
        model=YOLO(str(checkpoint))
    else:
        tag=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        run=root/'03_training_runs'/('equipment26_min100_round1_'+tag)
        run.mkdir(parents=True,exist_ok=False)
    status={'state':'preparing','run':str(run),'dataset_manifest_sha256':expected,'classes':26,'epochs_requested':100,'epochs_completed':0,'train_images':3128,'val_images':204,'test_images':205,'model_retrained':False}
    if resume:status.update(saved)
    def update(**values):
        status.update(values);status['updated_utc']=datetime.now(timezone.utc).isoformat()
        write_json(run/'status.json',status)
    try:
        cfg=yaml.safe_load((data/'data.yaml').read_text());cfg['path']=str(data)
        cfg_path=data/'data_train.yaml';cfg_path.write_text(yaml.safe_dump(cfg,sort_keys=False))
        for name in ['summary.json','class_mapping.json','selection_criteria_ko.txt']:
            shutil.copy2(data/name,run/name)
        shutil.copy2(Path(__file__),run/'train_min100.py')
        shutil.copy2(cfg_path,run/'data_train.yaml')
        if not resume:write_json(run/'initialization.json',{'checkpoint':str(source),'sha256':sha(source),'method':'fine-tune existing 26-class weights on expanded dataset; new optimizer, not resume old dataset'})
        write_json(run/'environment.json',{'python':sys.version,'torch':torch.__version__,'ultralytics':ultralytics.__version__,'gpu':torch.cuda.get_device_name(0)})
        def started(trainer):
            assert trainer.model.names==names
            update(state='training',gpu=torch.cuda.get_device_name(0))
            print('TRAINING_STARTED_26_CLASSES_MIN100',str(run),flush=True)
        def saved_epoch(trainer):
            update(state='training',epochs_completed=int(trainer.epoch+1),latest_validation_metrics={k:float(v) for k,v in trainer.metrics.items()},best_checkpoint=str(trainer.best),last_checkpoint=str(trainer.last))
            print('EPOCH_CHECKPOINT_SAVED',trainer.epoch+1,flush=True)
        model.add_callback('on_train_start',started)
        model.add_callback('on_model_save',saved_epoch)
        update(state='preparing');print('RUN_DIRECTORY',str(run),flush=True)
        if resume:model.train(resume=True,device=0,workers=2)
        else:model.train(data=str(cfg_path),epochs=100,patience=20,imgsz=640,batch=16,device=0,workers=2,seed=1398,deterministic=True,cache=False,fraction=1.0,val=True,save=True,save_period=-1,project=str(run),name='fit',exist_ok=False,plots=True,verbose=False,close_mosaic=10)
        best=run/'fit/weights/best.pt';assert best.is_file()
        update(state='evaluating_test',training_completed=True,model_retrained=True,best_checkpoint_sha256=sha(best))
        del model;gc.collect();torch.cuda.empty_cache()
        final=YOLO(str(best))
        metrics=final.val(data=str(cfg_path),split='test',imgsz=640,batch=16,device=0,workers=2,plots=True,save_json=False,project=str(run),name='test_evaluation',exist_ok=True,verbose=False)
        indices={int(c):i for i,c in enumerate(metrics.box.ap_class_index)};rows=[]
        for m in mapping:
            c=m['yolo_id'];i=indices.get(c);counts=summary['class_image_counts'][m['name']]
            row={**m,**counts,'small_test_sample':counts['test']<20}
            row.update({k:float(arr[i]) if i is not None else 0.0 for k,arr in [('precision',metrics.box.p),('recall',metrics.box.r),('ap50',metrics.box.ap50),('ap50_95',metrics.box.ap)]})
            rows.append(row)
        report={'images':205,'classes':26,'precision':float(metrics.box.mp),'recall':float(metrics.box.mr),'mAP50':float(metrics.box.map50),'mAP50_95':float(metrics.box.map),'per_class':rows,'field_validated':False,'limitations':[x for x in summary['limitations'] if x!='No retraining performed for this update.']}
        write_json(run/'test_metrics.json',report)
        update(state='completed',evaluation_completed=True,best_checkpoint=str(best),test_metrics={k:report[k] for k in ['precision','recall','mAP50','mAP50_95']})
        print('TRAINING_AND_TEST_COMPLETE',json.dumps(status,ensure_ascii=False),flush=True)
    except BaseException as exc:
        update(state='interrupted' if isinstance(exc,KeyboardInterrupt) else 'failed',error=str(exc),traceback=traceback.format_exc());raise
if __name__=='__main__':main()


In [ ]:
# Drive에 체크포인트를 직접 저장하며 이전 체크포인트에서 재학습합니다.
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, shutil, subprocess, sys, torch
root = Path('/content/drive/MyDrive/SKKU_1398_Fitness_ML')
data = Path('/content/equipment26_min100_v2')
assert shutil.disk_usage(root).free > 300*1024**2, 'Drive에 최소 300MB 여유가 필요합니다.'
archive = root/'02_dataset/SKKU1398_equipment_v1/yolo/equipment26_min100_v2_update.zip'
assert hashlib.sha256(archive.read_bytes()).hexdigest() == '0356d013ba2d8cbca0b511150665350f6c7e76f2ca2544f0f9490add75a42ee2'
if not (data/'manifest.json').is_file():
    subprocess.run([sys.executable, '/content/prepare_min100.py'], check=True)
old = root/'03_training_runs/equipment26_min100_round1_20261031T182137Z'
ckpt = torch.load(old/'fit/weights/last.pt', map_location='cpu', weights_only=False)
completed = int(ckpt['epoch'])+1
assert 0 < completed < 100 and ckpt.get('optimizer') is not None
print('RESUME_CHECKPOINT_COMPLETED_EPOCHS', completed, flush=True)
del ckpt
run = root/'03_training_runs'/('equipment26_min100_retrain_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
(run/'fit/weights').mkdir(parents=True, exist_ok=False)
for name in ['best.pt','last.pt']:
    shutil.copy2(old/'fit/weights'/name,run/'fit/weights'/name)
if (old/'fit/results.csv').is_file(): shutil.copy2(old/'fit/results.csv',run/'fit/results.csv')
status = {'state':'preparing','run':str(run),'dataset_manifest_sha256':'3faea06d469f471e85cd2dd976a55049cafcb9db5e4d96809e42a923cb754706','classes':26,'epochs_requested':100,'epochs_completed':completed,'train_images':3128,'val_images':204,'test_images':205,'model_retrained':False,'resumed_from':str(old)}
(run/'status.json').write_text(json.dumps(status,ensure_ascii=False,indent=2))
script = Path('/content/train_min100.py')
s = script.read_text().replace('model.train(resume=True,device=0,workers=2)', "model.train(resume=True,device=0,workers=2,save_dir=str(run/'fit'))")
compile(s,str(script),'exec')
script.write_text(s)
os.environ['EQUIPMENT26_MIN100_RESUME_RUN'] = str(run)
print('PERSISTENT_DRIVE_RUN',str(run),flush=True)
%run /content/train_min100.py


In [ ]:
# Convert the verified final checkpoint; keep original training files unchanged.
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, subprocess, sys
source_run = Path('/content/drive/MyDrive/SKKU_1398_Fitness_ML/03_training_runs/equipment26_min100_retrain_20261101T020110Z')
source = source_run/'fit/weights/best.pt'
assert hashlib.sha256(source.read_bytes()).hexdigest() == '7af2ceb79edad610f40369bdac110f2aeb80fae40c2921ec95a70eb081bc585b'
export_dir = source_run.parent.parent/'04_exports_and_reports'/('equipment26_tflite_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
export_dir.mkdir(parents=True, exist_ok=False)
work = Path('/content/equipment26_tflite_export'); work.mkdir(exist_ok=True)
shutil.copy2(source, work/'equipment26_fp32.pt')
shutil.copy2(source_run/'class_mapping.json', export_dir/'class_mapping.json')
code = """
from pathlib import Path
import json, shutil, hashlib, sys
from ultralytics import YOLO
import ultralytics, torch
work=Path('/content/equipment26_tflite_export')
out=Path(sys.argv[1])
model=YOLO(str(work/'equipment26_fp32.pt'))
assert len(model.names)==26
print('EXPORT_START',ultralytics.__version__,torch.__version__,flush=True)
exported=Path(model.export(format='litert',imgsz=640,batch=1,device='cpu',nms=False))
assert exported.is_file() and exported.suffix=='.tflite'
target=out/'equipment26_fp32.tflite'
shutil.copy2(exported,target)
(out/'labels.txt').write_text('\n'.join(model.names[i] for i in range(26))+'\n',encoding='utf-8')
report={'model':target.name,'bytes':target.stat().st_size,'sha256':hashlib.sha256(target.read_bytes()).hexdigest(),'ultralytics':ultralytics.__version__,'torch':torch.__version__,'source_sha256':'7af2ceb79edad610f40369bdac110f2aeb80fae40c2921ec95a70eb081bc585b','nms_included':False}
(out/'conversion_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('TFLITE_SAVED',str(target),json.dumps(report),flush=True)
"""
script=work/'export_model.py'; script.write_text(code)
shutil.copy2(script,export_dir/script.name)
print('EXPORT_DIRECTORY',str(export_dir),flush=True)
subprocess.run([sys.executable,str(script),str(export_dir)],check=True)


In [ ]:
from pathlib import Path
import subprocess,sys,shutil
work=Path('/content/equipment26_tflite_export')
export_dir=Path('/content/drive/MyDrive/SKKU_1398_Fitness_ML/04_exports_and_reports/equipment26_tflite_20261101T034127Z')
code=r'''
from pathlib import Path
import json,shutil,hashlib,sys
from ultralytics import YOLO
import ultralytics,torch
work=Path('/content/equipment26_tflite_export')
out=Path(sys.argv[1])
model=YOLO(str(work/'equipment26_fp32.pt'))
assert len(model.names)==26
print('EXPORT_START',ultralytics.__version__,torch.__version__,flush=True)
exported=Path(model.export(format='litert',imgsz=640,batch=1,device='cpu',nms=False))
assert exported.is_file() and exported.suffix=='.tflite'
target=out/'equipment26_fp32.tflite'
shutil.copy2(exported,target)
(out/'labels.txt').write_text(chr(10).join(model.names[i] for i in range(26))+chr(10),encoding='utf-8')
report={'bytes':target.stat().st_size,'sha256':hashlib.sha256(target.read_bytes()).hexdigest(),'ultralytics':ultralytics.__version__,'torch':torch.__version__,'nms_included':False}
(out/'conversion_report.json').write_text(json.dumps(report,indent=2))
print('TFLITE_SAVED',str(target),json.dumps(report),flush=True)
'''
script=work/'export_model.py'
compile(code,str(script),'exec')
script.write_text(code)
shutil.copy2(script,export_dir/script.name)
proc=subprocess.Popen([sys.executable,str(script),str(export_dir)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with (export_dir/'conversion_log.txt').open('w') as log:
    for line in proc.stdout:
        print(line,end='',flush=True)
        log.write(line);log.flush()
print('CONVERTER_EXIT_CODE',proc.wait())


In [ ]:
# Inspect actual tensor contract and evaluate both formats identically.
validation_code=r'''
from pathlib import Path
import sys,json,hashlib,zipfile,shutil,time
import numpy as np,torch,cv2
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox
from ai_edge_litert.interpreter import Interpreter
torch.set_num_threads(2)
out=Path(sys.argv[1]); work=Path('/content/equipment26_tflite_export')
f=out/'equipment26_fp32.tflite'
interpreter=Interpreter(model_path=str(f),num_threads=2)
interpreter.allocate_tensors()
def describe(d):
    return {'name':d['name'],'index':int(d['index']),'shape':d['shape'].tolist(),'dtype':np.dtype(d['dtype']).name,'quantization':[float(d['quantization'][0]),int(d['quantization'][1])]}
inputs=interpreter.get_input_details(); outputs=interpreter.get_output_details()
spec={'inputs':[describe(d) for d in inputs],'outputs':[describe(d) for d in outputs],'nms_included':False}
(out/'input_output_spec.json').write_text(json.dumps(spec,indent=2))
print('TENSOR_SPEC',json.dumps(spec),flush=True)
assert tuple(inputs[0]['shape'])==(1,3,640,640)
assert tuple(outputs[0]['shape'])==(1,30,8400)
assert inputs[0]['dtype']==np.float32
pt=YOLO(str(work/'equipment26_fp32.pt'))
pt.model.float().eval()
images=sorted(Path('/content/equipment26_min100_v2/images/test').glob('*'))
assert len(images)==205
checks=[]
for p in images[:5]:
    im=cv2.imread(str(p)); im=LetterBox(new_shape=(640,640),auto=False)(image=im)
    x=np.ascontiguousarray(im[:,:,::-1].transpose(2,0,1)[None],dtype=np.float32)/255.0
    interpreter.set_tensor(inputs[0]['index'],x); interpreter.invoke(); actual=interpreter.get_tensor(outputs[0]['index'])
    with torch.inference_mode(): expected=pt.model(torch.from_numpy(x))
    if isinstance(expected,(tuple,list)): expected=expected[0]
    expected=expected.numpy(); expected[:,:4,:]/=640.0
    checks.append({'image':p.name,'max_abs_error':float(np.max(np.abs(actual-expected))),'mean_abs_error':float(np.mean(np.abs(actual-expected)))})
print('RAW_PARITY',json.dumps(checks),flush=True)
report={'test_images':205,'raw_output_comparison':checks,'device':'Colab CPU; not a phone','validation_settings':{'imgsz':640,'batch':1,'rect':False,'conf':0.001,'iou':0.7},'models':{}}
for name,path in [('pytorch',work/'equipment26_fp32.pt'),('tflite',f)]:
    print('EVALUATE',name,flush=True)
    m=YOLO(str(path),task='detect').val(data='/content/equipment26_min100_v2/data_train.yaml',split='test',imgsz=640,batch=1,rect=False,device='cpu',workers=0,plots=False,save_json=False,verbose=False,conf=0.001,iou=0.7,project=str(work/'validation'),name=name,exist_ok=True)
    report['models'][name]={'precision':float(m.box.mp),'recall':float(m.box.mr),'mAP50':float(m.box.map50),'mAP50_95':float(m.box.map),'speed_ms':{k:float(v) for k,v in m.speed.items()}}
    (out/'validation_report.json').write_text(json.dumps(report,indent=2))
    print('METRICS',name,json.dumps(report['models'][name]),flush=True)
print('VALIDATION_COMPLETE',json.dumps(report),flush=True)
'''
validation_script=work/'validate_tflite.py';validation_script.write_text(validation_code)
shutil.copy2(validation_script,export_dir/validation_script.name)
proc=subprocess.Popen([sys.executable,str(validation_script),str(export_dir)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with (export_dir/'validation_log.txt').open('w') as log:
    for line in proc.stdout:
        print(line,end='',flush=True);log.write(line);log.flush()
print('VALIDATOR_EXIT_CODE',proc.wait())


In [ ]:
# Only upload the exact model and build script; no Drive mount.
from google.colab import files
uploaded=files.upload()
print('UPLOADED_FILES',list(uploaded))
